<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-HUYTRUONG/AIO/blob/aio/M4-Foundation-DL/Optimization/W1/Logistic%20Regression/01_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- **Logistic Regression** là một thuật toán **học máy giám sát (supervised learning)** được sử dụng phổ biến trong các bài toán **phân loại nhị phân (binary classification)**. Mặc dù có tên gọi là Regression, nhưng thuật toán này không nằm trức tiếp dự đoán một giá trị liên tục như **Linear Regression**, mà ước lượng: *"một quan sát thuộc về một trong hai lớp"*.

- Tài liệu này sẽ trình bày về nền tảng toán học của **Logisitc Regression** thông qua các thành phần chính:
  - Mô hình tuyến tính.
  - Hàm Sigmoid.
  - Hàm mất mát **Binary Cross-Entropy (BCE)**.
  - Và thuật toán tối ưu Gradient Descent.

- Mục tiêu của tài liệu này là thiết lập một cách hiểu nhất về **Logisitc Regression**:

$$
\boxed{
\text{Input} \to \text{Sigmoid} \to \text{Prediction} \to \text{BCE} \to \text{Gradient} \to \text{Update}
}
$$


# **1. Vì sao chúng ta lại cần Logistic Regression**



## **1.1. Bắt đầu từ Linear Regression**
- Giả sử đâu vào $x$ là diện tích nhà, đâu ra $y$ là giá nhà. Một mô hình tuyến tính có dạng: $$\hat{y} = wx + b$$

- Trong đó:
  - $x$: đặc trưng đầu vào (feature), chẳng hạn diện tích.
  - $w$: trọng số (weight), quyết định mức độ đầu ra thay đổi theo đầu vào.
  - $b$: hệ số chệch (bias).
  - $\hat y$: giá trị mô hình dự đoán.
  - $y$: giá trị thực tế (label).

- Mô hình sẽ tìm các giá trị $w, b$ sao cho dự đoán gần với dữ liệu thức tế. VD: $$\hat{y} = 2x + 3$$

  - Nếu $x = 4$, mô hình dự đoán: $\hat{y} = 2(4) + 3 = 11$. Đầu ra của phương trình tuyến có thể nằm trong khoảng từ âm vô cùng đến dướng vô cùng: $$\hat y\in(-\infty,+\infty)$$
  
  - Điều này phù hợp với nhiều bài toán hồi quy, nhưng lại gặp vấn đề khi ta muốn phân loại dữ liệu thành 2 nhóm.

## **1.2. Chuyển từ dự đoán giá trị sang phân loại**

- Hãy xét một bài toán khác: Dự đoán email có phải spam hay không. Chúng ta sẽ mã hóa nhãn thành hai giá trị:
  - $ y = 0 $: Không phải spam.
  - $ y = 1 $: Spam.

- Lúc này, mục tiêu không cần là dự đoán một giá trị bất kỳ nữa, chúng ta cần xác định dữ liệu thuộc nhóm nào. Nếu dùng $\hat{y} = wx +b$ - Mô hình có thể tạo ra $\hat{y} = 2.7, \hat{y} = -1.3$, hoặc bất kì giá trị nào khác. Những kết quả này không được dễ diễn giải trực tiếp thành xác suất thuộc lớp.

- Chúng ta có thể dùng **ngưỡng (threshold)** để phân loại, nhưng điều đó chưa giải quyết được vấn đề chúng ta phải làm việc với $$\hat y\in(-\infty,+\infty)$$

- **Vấn đề cốt lõi ở đây**: Phương trình tuyến tính không đảm bảo đầu ra nằm trong khoảng $[0,1]$ - Trong khi ta muốn biển diễn mực độ tin tưởng của trong khoảng $[0,1]$. Đây là lúc chúng ta dùng **Sigmoid** để giải quyết.

# **2. Sigmoid Function - Biến đầu ra thành xác suất**

- Thay vì sử dụng trực tiếp đầu ra của phương trình tuyến tính, ta đưa đầu ra đó qua một hàm có khả năng né mội số thực vào khoảng $[0,1]$.

- **Sigmoid**:
  $$\sigma(z)=\frac{1}{1+e^{-z}} \quad \sigma(z) \in (0,1) $$

- Với **Logistic Regression**:
  $$z = wx + b \quad z \in(-\infty,+\infty)$$

- **Sigmoid** là hàm tăng: $z$ càng lớn thì $\sigma(z)$ càng lớn. Tuy nhiên, đầu ra luôn nằm trong khoảng $(0,1)$, không chạm chính xác 0 hoặc 1 với đầu vào hữu hạn.

## **2.1. Từ xác suất đến nhãn dữ đoán**

- Giả sử mô hình dự đoán: $\hat{y} = 0.92$. Ta có thể diễn giải rằng mô hình ước lượng xác suất thuộc lớp 1 là 92%, theo cách diễn giải của Logistic Regression** nhị phân.

- Nếu sử dụng `threshold = 0.5` quy tắc phân loại là:
$$
\hat c=
\begin{cases}
1 & \text{nếu }\hat y\geq 0.5\\
0 & \text{nếu }\hat y<0.5
\end{cases}
$$

- `threshold = 0.5` là lựa chọn phổ biến, không phải ngưỡng bắt buộc trong mọi tình huống.

- **Điểm mấu chốt**: **Sigmoid** không tự học các tham số. Nó chuyển đầu ra tuyến tính thành giá trị phù hợp hơn cho bài toán phân loại. Việc học $w, b$ sẽ được thực hiện thông qua hàm **Loss** và **Gradient Descent**

# **3. Loss Function - Làm sao biết mô hình dự đoán tốt hay tệ**

- Sau khi mô hình tạo ra dự đoán $\hat{y}$, ta cần phải so sánh nó với nhãn thực tế là $y$.
- VD: Cả hai trường hợp sau đều có nhãn thực tế $y = 1$:
  - Mô hình A dự đoán $\hat{y} = 0.98$
  - Mô hình B dự đoán $\hat{y} = 0.14$

  - **Mô hình A thể hiện sự tin tưởng cao và lớp đúng, trong khi mô hình B** nghiên về lớp sai. Câu hỏi đặt ra là: **Làm thế nào để biết dự đoán của mô hình tốt hay tệ?** - Ta cần một cách đo lường. Đó chính là vai trò của Loss.

- Loss giúp ta **chuyển mức độ phù hợp của một dự đoán thành một con số có thể đo lường**:
  - Mô hình A dự đoán với 0.98% gần với dự liệu thực tế $y = 1$ => Loss thấp - dự đoán phù hợp với thực tế

  - Mô hình B dự đoán với 0.14% xa với dữ liệu thức tế $y = 1% => Loss cao - mô hình đang nghiêng về dự đoan sai.

# **4. Binary Cross-Entropy**

- Với bài toán phân loại nhị phân, nhãn $y$ chỉ nhận một trong hai giá trị 0 hoặc 1. Ta sẽ bắt đầu với 2 trường hợp

## **4.1. Nhãn thực tế $y = 1$**

- Mô hình cần dự đoán xác suất thuộc top 1 càng cao càng tốt. Loss được xác định:

$$ L=-\log(\hat y) $$

  - Nếu $\hat{y} = 0.9$, Loss nhỏ: $ L= - \log(0.9)\approx 0.105 $
  - Nếu $\hat{y} = 0.1$, Loss lớn: $ L= - \log(0.1)\approx 2.303 $

## **4.2 Nhãn thực tế $y = 0$**

- Lúc này, mô hình cần dự đoán xác suất thuộc lớp 1 thấp. Loss được xác định bằng:
$$ L=-\log(1-\hat y) $$

  - Nếu $\hat y=0.1$, Loss nhỏ: $L=-\log(0.9)\approx 0.105 $
  - Nếu $\hat y=0.9$, Loss lớn: $L=-\log(0.1)\approx 2.303 $

## **4.3. Gộp hai trường hợp thành một công thức**

- Ta có thể kết hợp cả hai công thức bằng cách sử dụng nhãn $y$:
$$ \boxed{
L(y,\hat y)=
-y\log(\hat y)
-(1-y)\log(1-\hat y)
}
$$

- Đây là **Binary Cross-Entropy (BCE)**. Vì $y$ chỉ bằng 0 hoặc 1:
  - Nếu $y=1$, biểu thức tự rút gọn thành $-\log(\hat y)$.
  - Nếu $y=0$, biểu thức tự rút gọn thành $-\log(1-\hat y)$.

# **5. Derivatives — Làm sao biết phải thay đổi tham số theo hướng nào?**

- Chúng ta đã biết:
  - Mô hình: $\hat y=\sigma(z)=\frac{1}{1+e^{-z}}$
  - Hàm loss: $L=-y\log\hat y-(1-y)\log(1-\hat y)$

- Nhưng biết Loss lớn chưa đủ. Ta còn cần biết nên điều chỉnh $w$ và $b$ như thế nào để Loss giảm xuống. Đây là vai trò của **đạo hàm (derivative)**.

## **5.1. Đạo hàm mang ý nghĩa gì?**

- Giả sử ta xem **Loss** như một hàm phụ thuộc vào \(w\).
$$ L=L(w) $$

- Đạo hàm: $ \frac{\partial L}{\partial w} $. cho biết **Loss** thay đổi theo hướng nào khi $w$ thay đổi một lượng nhỏ, trong khi những yếu tố còn lại được giữ cố định.
  - Đạo hàm dương: tăng $w$ một chút có xu hướng làm Loss tăng.
  - Đạo hàm âm: tăng $w$ một chút có xu hướng làm Loss giảm.
  - Đạo hàm bằng 0: độ dốc tại điểm đó bằng 0 - đây có thể là điểm cực trị hoặc một điểm dừng khác.

- Tương tự, ta cần tính $\frac{\partial L}{\partial b}$ để biết cách điều chỉnh bias.

## **5.2. Tại sao cần Chain Rule?**

- Loss không phụ thuộc trực tiếp vào \(w\) theo một bước đơn giản. Nó phụ thuộc qua nhiều phép biến đổi:
$$
w\longrightarrow z=wx+b
\longrightarrow \hat y=\sigma(z)
\longrightarrow L(y,\hat y)
$$

- Vì vậy, ta sử dụng **Chain Rule** — quy tắc đạo hàm hàm hợp:

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial \hat y}
\frac{\partial \hat y}{\partial z}
\frac{\partial z}{\partial w}
$$

- Với **BCE** và **Sigmoid**, các thành phần lần lượt là:

$$ \frac{\partial L}{\partial\hat y}
=-\frac{y}{\hat y}+\frac{1-y}{1-\hat y} $$

$$ \frac{\partial\hat y}{\partial z}
=\hat y(1-\hat y) $$

$$ \frac{\partial z}{\partial w}=x $$

- Khi nhân và rút gọn, ta thu được kết quả rất quan trọng:
$$ \boxed{\frac{\partial L}{\partial w}=x(\hat y-y)} $$

- Đối Với bias:
$$ \boxed{\frac{\partial L}{\partial b}=\hat y-y} $$

- Trong đó:
  - $\hat y-y$: độ lệch giữa dự đoán và nhãn thực tế.
  - $x$: mức độ đầu vào ảnh hưởng đến gradient của trọng số $w$.

- Ví dụ, nếu: $$ x=2,\quad y=1,\quad\hat y=0.7 $$

  - $$ \frac{\partial L}{\partial w} =2(0.7-1)=-0.6 $$

  - $$ \frac{\partial L}{\partial b}=0.7-1=-0.3 $$

- Cả hai đạo hàm đều âm. Theo quy tắc cập nhật **Gradient Descent** ở phần tiếp theo, các tham số sẽ được điều chỉnh theo hướng tăng $w$ và $b$ trong bước này.

# **6. Gradient Descent — Cách mô hình học từ dữ liệu**

- Ta có hai tham số cần học: $w$ và $b$. **Gradient Descent** sẽ giúp ta cập nhật chúng theo quy tắc:

$$
\boxed{w_{\text{new}}=w-\eta\frac{\partial L}{\partial w}}
$$

$$
\boxed{b_{\text{new}}=b-\eta\frac{\partial L}{\partial b}}
$$

- Trong đó:
  - $\eta$ là **learning rate**: tốc độ học, quyết định độ lớn của bước cập nhật.
  - Dấu trừ rất quan trọng: **Gradient Descent** di chuyển ngược hướng **gradient** để tìm cách làm giảm **Loss**.

- Giả sử:
$$ x=2,\quad y=1,\quad w=0,\quad b=0 \quad learning rate:
\eta=0.1 $$

- **Bước 1 — Tính đầu ra tuyến tính**: $ z = wx + b = 0(2) + 0 = 0 $
- **Bước 2 — Tính xác suất dự đoán**: $\hat y = \sigma(0) = 0.5 $
- **Bước 3 — Tính BCE**: Vì nhãn thực tế $ y = 1 $: $ L=-\log(0.5)\approx0.6931 $
- **Bước 4 — Tính đạo hàm**: $$ \frac{\partial L}{\partial w}
=x(\hat y-y)=2(0.5-1)=-1
$$

$$
\frac{\partial L}{\partial b}
=\hat y-y=0.5-1=-0.5
$$

- **Bước 5 — Cập nhật tham số**:
$$
w_{\text{new}}=0-0.1(-1)=0.1
$$

$$
b_{\text{new}}=0-0.1(-0.5)=0.05
$$

- **Bước 6 - Sau bước cập nhật, mô hình có**:
$$
w=0.1,\qquad b=0.05
$$

$$ z=0.1(2)+0.05=0.25 $$

$$ \hat y=\sigma(0.25)\approx0.5622 $$

- Xác suất dự đoán lớp 1 đã tăng từ $0.5$ lên khoảng $0.5622$, phù hợp với nhãn thực tế $y=1$. BCE của mẫu này cũng giảm từ khoảng $0.6931$ xuống khoảng $0.5762$.